> **m2mTani-MT: fine-tuning notebook**
>
> This is the complete notebook used to train [m2mTani-MT](https://huggingface.co/YOUR_HF_USERNAME/m2mTani-MT)
> ([demo Space](https://huggingface.co/spaces/YOUR_HF_USERNAME/m2mTani-MT-app)). The released checkpoint is
> `m2m-tani-nllb-32k-steps` (32,000 optimizer steps).
>
> **Data.** The full corpus is not distributed (most of it is Bible text, which belongs to its publishers). Put
> your corpus in `CORPORA` (cell 1) in this layout:
>
> ```
> mtani-corpora/
> ├── Adi/adi_english_parallel.csv            columns: adi, english, source
> ├── Apatani/apatani_english_parallel.csv    columns: apatani, english, source
> ├── Galo/galo_english_parallel.csv          columns: galo, english, source
> ├── Hindi/hindi_english_parallel.csv        columns: hindi, english, source
> ├── Nyishi/nyishi_english_parallel.csv      columns: nyishi, english, source
> ├── Tagin/tagin_english_parallel.csv        columns: tagin, english, source
> └── tani_corpus.json                        Bible verse references: {"entries": [{"lang", "text", "domain", "group": "bible:<BOOK c:v>"}, ...]}
> ```
>
> One row is one (source text, English translation) pair. A source text may appear in many rows, one per English
> version. `tani_corpus.json` is only used to find which rows are Bible verses (and which verse), so that the split
> is by chapter and Tani ↔ Tani pairs can be built. Rows not found there are treated as non-Bible text. See
> [`data/README.md`](../data/README.md) for the Bible sources and the 2,000-pair non-Bible sample
> (`data/sample_2k_non_bible.parquet`), which can be written out in this layout for a quick smoke test.
>
> **Citation.** Tungon Dugi, *m2m-Tani-MT: Finetuning Multilingual Machine Translation Model for 5 Tani languages of
> Arunachal Pradesh*, Department of Computer Science and Engineering, National Institute of Technology. Manuscript
> under peer review.

# Many-to-many NMT for five Tani languages, Hindi and English: a controlled experiment

**Question.** How well does `facebook/nllb-200-distilled-600M`, fine-tuned on the Tani corpora, translate in
**all 42 directions** among Tagin, Nyishi, Apatani, Galo, Adi (languages NLLB-200 has never seen), Hindi and
English, and does the fine-tuning keep NLLB's existing Hindi ↔ English intact?

**Protocol (fixed before training).**

| | |
|---|---|
| Data | `../mtani-corpora`: each language paired with English (≈3.0M pairs, 330K source texts). Pairs between non-English languages come from the same Bible verse in both languages (verse references from `tani_corpus.json`): ≈7K verses for every pair, ≈30K among Adi, Nyishi and Hindi |
| Unit of splitting | Bible: whole **chapters**, identical across all languages, so a verse never appears in two splits in any language or pair. Other text: groups of source texts linked by a shared English translation |
| Split | **90 / 5 / 5** train / dev / test (dev selects the checkpoint; test is scored once, at the end) |
| Leakage audit | A held-out item is flagged if its English occurs anywhere in training, or if a source text in it nearly matches a training text (3-gram consonant-skeleton Jaccard ≥ 0.8, any spelling); headline scores use **clean** items only |
| Model | NLLB-200 600M + five new language codes (`tgj_Latn`, `njz_Latn`, `apt_Latn`, `adl_Latn`, `adi_Latn`), each initialised from the mean of `lus_Latn` and `kac_Latn` |
| Training data per epoch | Every source text with up to 3 of its English versions, both directions; language temperature α = 0.3; synthetic (KIFS) rows weight 0.5. Plus, for every training verse found in 2 or more non-English languages, up to 3 random language pairs, both directions |
| Optimisation | Adafactor, lr 1e-4, 2,000 warm-up steps, inverse-sqrt decay, label smoothing 0.1, bf16, 16K tokens/step |
| Stopping and selection | Dev chrF++ (greedy, 100 items per direction) averaged over all 42 directions every 2,000 steps; the best checkpoint is kept; **early stopping** after 3 evaluations in a row without improvement. The untrained model (step 0) is the first best checkpoint |
| Metrics | chrF++ (primary) and BLEU (sacreBLEU, signatures recorded) with 95% bootstrap confidence intervals (1,000 resamples), every direction; English directions also by domain |
| Baselines | Step-0 model on dev; untouched NLLB-200 on Hindi ↔ English test |
| Reproducibility | Fixed seed; config, environment, file checksums, split manifest, leakage audit and every prediction saved under `OUT` |

Pairs between non-English languages are Bible text only: the everyday, dictionary and KIFS data is paired with
English only.

**Hardware.** One NVIDIA RTX 4000 Ada (20 GB). Training runs until early stopping; final evaluation adds one to
two hours. If the kernel stops, run all cells again: data preparation is cached and training resumes from the
last checkpoint. For an unattended run:

```bash
jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 \
    --output experiment_run.ipynb nllb_tani_experiment.ipynb
```

## 1. Settings

In [ ]:
from pathlib import Path

CORPORA = Path("../mtani-corpora")                       # <Lang>/<lang>_english_parallel.csv and tani_corpus.json
TANI_JSON = CORPORA / "tani_corpus.json"                 # Bible verse references
MODEL = "facebook/nllb-200-distilled-600M"
OUT = Path("runs/nllb600M-tani-m2m-90-5-5")
SEED = 13

# split
SPLIT_FRACTIONS = {"train": 0.90, "dev": 0.05, "test": 0.05}
NEAR_DUP_THRESHOLD = 0.8          # skeleton 3-gram Jaccard at or above this counts as a near-duplicate
MAX_COMPONENT = 2000              # larger groups of texts linked by shared English always go to train
REF_PRIORITY = ["original", "engbsb", "engwebp", "eng-asv"]   # English version used as the single reference
HINDI_PRIMARY = "hin2017"         # Hindi translation used when a verse is source or reference in evaluation

# training data
K_ENGLISH = 3                     # English versions sampled per source text per epoch
ALPHA = 0.3                       # language sampling temperature (1 = proportional to size)
SYNTHETIC_WEIGHT = 0.5            # sampling weight of machine-generated (KIFS) rows
XX_PAIRS_PER_VERSE = 3            # non-English language pairs sampled per multilingual verse per epoch
BUCKETS = [24, 32, 48, 64, 80, 96, 112, 128]   # padded lengths; longer pairs are skipped

# optimisation
TOKENS_PER_BATCH = 2048           # lower it if the GPU runs out of memory (and raise ACCUM)
ACCUM = 8
LR = 1e-4
WARMUP = 2000
LABEL_SMOOTHING = 0.1
INIT_FROM = ["lus_Latn", "kac_Latn"]
LOG_EVERY = 100
SAVE_MINUTES = 30

# stopping and evaluation
EVAL_EVERY_STEPS = 2000
PATIENCE = 3                      # stop after this many dev evaluations in a row without improvement
MIN_DELTA = 0.0                   # chrF++ gain needed to count as an improvement
DEV_EVAL_PER_DIRECTION = 100      # fixed dev sample per direction, greedy decoding
TEST_BEAM = 4
TEST_MAX_PER_DIRECTION = 1000     # seeded sample of clean test items per direction (None = all)
EVAL_BATCH = 48
BOOTSTRAP = 1000
BASELINE_HINDI = True             # also score the untouched NLLB-200 on Hindi <-> English test

## 2. Environment, seeds and checksums

In [ ]:
import hashlib, itertools, json, platform, random, sys, time
import numpy as np
import torch, transformers, sacrebleu

assert torch.cuda.is_available(), "No CUDA GPU found: this notebook is written for the RTX 4000 Ada."
DEVICE = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

OUT.mkdir(parents=True, exist_ok=True)
LANGS = {"Tagin": ("tgj_Latn", "tagin"), "Nyishi": ("njz_Latn", "nyishi"), "Apatani": ("apt_Latn", "apatani"),
         "Galo": ("adl_Latn", "galo"), "Adi": ("adi_Latn", "adi"), "Hindi": ("hin_Deva", "hindi")}
ENG = "eng_Latn"
ALL_CODES = [c for c, _ in LANGS.values()] + [ENG]
CODE2NAME = {**{c: n for n, (c, _) in LANGS.items()}, ENG: "English"}
DIRECTIONS = [(a, b) for a in ALL_CODES for b in ALL_CODES if a != b]          # 42


def sha256(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()


env_path = OUT / "environment.json"
if not env_path.exists():
    files = [TANI_JSON] + [CORPORA / n / f"{c}_english_parallel.csv" for n, (_, c) in LANGS.items()]
    env = {"python": sys.version.split()[0], "platform": platform.platform(), "torch": torch.__version__,
           "cuda": torch.version.cuda, "transformers": transformers.__version__, "sacrebleu": sacrebleu.__version__,
           "gpu": torch.cuda.get_device_name(0), "started": time.strftime("%Y-%m-%d %H:%M:%S"),
           "sha256": {str(p): sha256(p) for p in files}}
    env_path.write_text(json.dumps(env, indent=2))
config = {k: (str(v) if isinstance(v, Path) else v) for k, v in globals().items() if k.isupper() and not k.startswith("_")
          and isinstance(v, (str, int, float, bool, list, dict, Path, type(None)))}
(OUT / "config.json").write_text(json.dumps(config, indent=2))
print(json.loads(env_path.read_text())["gpu"], "|", torch.__version__, transformers.__version__, sacrebleu.__version__)
print(f"{len(DIRECTIONS)} translation directions")

## 3. Load the corpora

Every distinct source text becomes a *group* with all of its English translations, tagged with its domain:
**bible** (verse found in `tani_corpus.json`), **synthetic** (KIFS) or **other** (dictionary examples,
conversation, news).

In [ ]:
import csv, re, unicodedata, collections

PUNCT = str.maketrans({"“": '"', "”": '"', "„": '"', "«": '"', "»": '"', "‘": "'",
                       "’": "'", "‚": "'", "–": "-", "—": " - ", "…": "..."})


def ekey(s):
    # English matching key: case, accents and punctuation ignored
    s = unicodedata.normalize("NFKD", s.casefold())
    return re.sub(r"[\W_]+", "", "".join(c for c in s if not unicodedata.combining(c)))


def tkey(s):
    # source matching key against tani_corpus.json (Tagin nyii was restored from nyi after it was built)
    return " ".join("nyi" if w == "nyii" else w for w in s.split())


G_lang, G_text, G_eng, G_src, G_synth = [], [], [], [], []
for name, (lc, col) in LANGS.items():
    idx = {}
    with open(CORPORA / name / f"{col}_english_parallel.csv", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            src, eng = row[col].translate(PUNCT).strip(), row["english"].translate(PUNCT).strip()
            if not src or not eng:
                continue
            g = idx.get(src)
            if g is None:
                g = idx[src] = len(G_text)
                G_lang.append(lc); G_text.append(src); G_eng.append([]); G_src.append([]); G_synth.append(False)
            G_eng[g].append(eng); G_src[g].append(row["source"])
            G_synth[g] |= "kifs-corpus" in row["source"] or "tgj-eng-dataset" in row["source"]
N = len(G_text)

verse_of = {}
with open(TANI_JSON, encoding="utf-8") as f:
    for e in json.load(f)["entries"]:
        if e.get("domain") == "bible" and str(e.get("group", "")).startswith("bible:"):
            verse_of[(e["lang"], tkey(e["text"].translate(PUNCT).strip()))] = e["group"][6:]
G_verse = [verse_of.get((G_lang[g], tkey(G_text[g]))) for g in range(N)]
G_chapter = [v.split(":")[0] if v else None for v in G_verse]
G_domain = ["bible" if G_verse[g] else "synthetic" if G_synth[g] else "other" for g in range(N)]
G_primary = [G_lang[g] != "hin_Deva" or any(HINDI_PRIMARY in s for s in G_src[g]) for g in range(N)]

summary = collections.Counter((CODE2NAME[G_lang[g]], G_domain[g]) for g in range(N))
print(f"{N:,} source texts, {sum(map(len, G_eng)):,} English pairs")
for name in LANGS:
    print(f"  {name:8}", "  ".join(f"{d} {summary[(name, d)]:>7,}" for d in ("bible", "synthetic", "other")))
verse_langs = collections.defaultdict(set)
for g in range(N):
    if G_verse[g]:
        verse_langs[G_verse[g]].add(G_lang[g])
pair_count = collections.Counter(p for ls in verse_langs.values() for p in itertools.combinations(sorted(ls), 2))
print("verses shared by each non-English pair:", ", ".join(f"{CODE2NAME[a]}-{CODE2NAME[b]} {n:,}" for (a, b), n in pair_count.most_common()))

## 4. Split

1. **Bible:** whole chapters are shuffled with the seed and assigned 90/5/5, the same chapters in every
   language, so verse-aligned pairs between any two languages fall in the same split.
2. **Other text:** a group that shares an English translation with Bible verses follows their split; if those
   verses sit in different splits it is dropped. The remaining groups are joined into components through
   shared English, and whole components are assigned 90/5/5.

In [ ]:
rng = np.random.default_rng(SEED)
split = [None] * N

chapters = sorted({c for c in G_chapter if c})
order = rng.permutation(len(chapters))
n_test = round(len(chapters) * SPLIT_FRACTIONS["test"]); n_dev = round(len(chapters) * SPLIT_FRACTIONS["dev"])
chapter_split = {chapters[i]: ("test" if k < n_test else "dev" if k < n_test + n_dev else "train") for k, i in enumerate(order)}
for g in range(N):
    if G_chapter[g]:
        split[g] = chapter_split[G_chapter[g]]

bible_split_of_eng = collections.defaultdict(set)
for g in range(N):
    if G_verse[g]:
        for e in set(map(ekey, G_eng[g])):
            bible_split_of_eng[e].add(split[g])
free, dropped = [], 0
for g in range(N):
    if split[g] is not None:
        continue
    linked = set().union(*(bible_split_of_eng.get(e, set()) for e in set(map(ekey, G_eng[g]))))
    if len(linked) == 1:
        split[g] = linked.pop()
    elif len(linked) > 1:
        split[g] = "dropped"; dropped += 1
    else:
        free.append(g)
parent = {g: g for g in free}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
first_with = {}
for g in free:
    for e in set(map(ekey, G_eng[g])):
        if e in first_with:
            a, b = find(g), find(first_with[e])
            if a != b:
                parent[a] = b
        else:
            first_with[e] = g
components = collections.defaultdict(list)
for g in free:
    components[find(g)].append(g)
comp_list = sorted(components.values(), key=lambda c: min(c))
for k in rng.permutation(len(comp_list)):
    comp, r = comp_list[k], rng.random()
    s = "train" if len(comp) > MAX_COMPONENT else "test" if r < SPLIT_FRACTIONS["test"] else "dev" if r < SPLIT_FRACTIONS["test"] + SPLIT_FRACTIONS["dev"] else "train"
    for g in comp:
        split[g] = s

counts = collections.Counter((CODE2NAME[G_lang[g]], split[g]) for g in range(N))
print(f"Bible chapters: {len(chapters):,} -> test {n_test}, dev {n_dev}; other text: {len(comp_list):,} components; dropped {dropped:,} conflicting groups")
print(f"{'':8} {'train':>8} {'dev':>7} {'test':>7} {'dropped':>8}   held-out share")
for name in LANGS:
    c = [counts[(name, s)] for s in ("train", "dev", "test", "dropped")]
    print(f"{name:8} {c[0]:>8,} {c[1]:>7,} {c[2]:>7,} {c[3]:>8,}   dev {c[1] / max(1, sum(c[:3])):.1%}, test {c[2] / max(1, sum(c[:3])):.1%}")

with open(OUT / "split_manifest.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["lang", "split", "domain", "verse", "source_sha1"])
    for g in range(N):
        w.writerow([G_lang[g], split[g], G_domain[g], G_verse[g] or "", hashlib.sha1(G_text[g].encode()).hexdigest()])
print(f"split manifest: {OUT / 'split_manifest.csv'}")

## 5. Leakage audit and evaluation sets

**English directions:** each dev or test group gives one item per direction, with one English reference (the
first available of `REF_PRIORITY`, otherwise the alphabetically first). **Other directions:** each held-out
verse present in both languages gives one item (Hindi uses `HINDI_PRIMARY`). An item is flagged when its
English occurs in training, or when a source text in it is a near-duplicate of a training text in any spelling
(Latin-script languages). Flagged items are left out of the headline (*clean*) scores.

In [ ]:
def skeleton(s):
    s = unicodedata.normalize("NFKD", s.casefold())
    s = "".join(c for c in s if not unicodedata.combining(c)).replace("ny", "x").replace("ng", "q")
    s = re.sub(r"[aeiouvwy]", "", re.sub(r"[^a-z]", "", s))
    return re.sub(r"(.)\1+", r"\1", s)


def grams(s):
    return {s[i:i + 3] for i in range(len(s) - 2)}


train_eng = {ekey(e) for g in range(N) if split[g] == "train" for e in G_eng[g]}
near_dup = set()
for lc in ALL_CODES:
    if not lc.endswith("_Latn") or lc == ENG:
        continue
    gr = {g: grams(skeleton(G_text[g])) for g in range(N) if G_lang[g] == lc and split[g] == "train"}
    gr = {g: s for g, s in gr.items() if len(s) >= 6}
    freq = collections.Counter(x for s in gr.values() for x in s)
    index = collections.defaultdict(list)
    for g, s in gr.items():
        for x in sorted(s, key=lambda x: (freq[x], x))[: len(s) - int(np.ceil(NEAR_DUP_THRESHOLD * len(s))) + 1]:
            index[x].append(g)
    for h in (g for g in range(N) if G_lang[g] == lc and split[g] in ("dev", "test")):
        A = grams(skeleton(G_text[h]))
        if len(A) < 6:
            continue
        probe = sorted(A, key=lambda x: (freq.get(x, 0), x))[: len(A) - int(np.ceil(NEAR_DUP_THRESHOLD * len(A))) + 1]
        for c in {c for x in probe for c in index.get(x, ())}:
            B = gr[c]; inter = len(A & B)
            if inter / (len(A) + len(B) - inter) >= NEAR_DUP_THRESHOLD:
                near_dup.add(h); break


def reference(g):
    by = {s.split("#")[-1].split(":")[-1]: e for e, s in zip(G_eng[g], G_src[g])}
    return next((by[v] for v in REF_PRIORITY if v in by), sorted(G_eng[g])[0])


eval_items = []
for g in range(N):
    if split[g] not in ("dev", "test") or not G_primary[g]:
        continue
    eng = reference(g)
    flags = {"eng_in_train": ekey(eng) in train_eng, "near_dup": g in near_dup}
    clean = not any(flags.values())
    eval_items.append({"split": split[g], "src_lang": G_lang[g], "tgt_lang": ENG, "src": G_text[g], "ref": eng, "domain": G_domain[g], "clean": clean, **flags})
    eval_items.append({"split": split[g], "src_lang": ENG, "tgt_lang": G_lang[g], "src": eng, "ref": G_text[g], "domain": G_domain[g], "clean": clean, **flags})
held_verse = collections.defaultdict(dict)                  # (split, verse) -> lang -> group
for g in range(N):
    if G_verse[g] and split[g] in ("dev", "test") and G_primary[g]:
        held_verse[(split[g], G_verse[g])].setdefault(G_lang[g], g)
for (s, v), by_lang in held_verse.items():
    for a, b in itertools.permutations(sorted(by_lang), 2):
        ga, gb = by_lang[a], by_lang[b]
        nd = ga in near_dup or gb in near_dup
        eval_items.append({"split": s, "src_lang": a, "tgt_lang": b, "src": G_text[ga], "ref": G_text[gb], "domain": "bible",
                           "clean": not nd, "eng_in_train": False, "near_dup": nd})

audit = collections.defaultdict(collections.Counter)
for it in eval_items:
    kind = "with English" if ENG in (it["src_lang"], it["tgt_lang"]) else "between other languages"
    a = audit[(it["split"], kind)]
    a["items"] += 1; a["eng_in_train"] += it["eng_in_train"]; a["near_dup"] += it["near_dup"]; a["clean"] += it["clean"]
print("leakage audit (source texts never cross splits by construction):")
print(f"{'':34} {'items':>8} {'English in train':>17} {'near-duplicate':>15} {'clean':>8}")
for (s, k), a in sorted(audit.items()):
    print(f"{s + ', ' + k:34} {a['items']:>8,} {a['eng_in_train']:>17,} {a['near_dup']:>15,} {a['clean']:>8,}")
(OUT / "leakage_audit.json").write_text(json.dumps({f"{s}/{k}": dict(a) for (s, k), a in audit.items()}, indent=2))

per_dir = collections.Counter((it["split"], it["src_lang"], it["tgt_lang"]) for it in eval_items if it["clean"])
print("clean test items per direction: min", min(per_dir[("test", a, b)] for a, b in DIRECTIONS), "max", max(per_dir[("test", a, b)] for a, b in DIRECTIONS))
drng = np.random.default_rng(SEED)
dev_sample = []
for a, b in DIRECTIONS:
    pool = [it for it in eval_items if it["split"] == "dev" and it["src_lang"] == a and it["tgt_lang"] == b and it["clean"]]
    if pool:
        dev_sample += [pool[i] for i in sorted(drng.choice(len(pool), size=min(DEV_EVAL_PER_DIRECTION, len(pool)), replace=False))]
print(f"dev sample for checkpoint selection: {len(dev_sample):,} translations over {len({(i['src_lang'], i['tgt_lang']) for i in dev_sample})} directions")

## 6. Training data cache

Only **train** groups are tokenized, together with the multilingual verses among them. The cache is stored
under `OUT` and reused on restart.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
CACHE = OUT / "train_cache.pt"


def ragged(lists):
    lengths = np.fromiter((len(x) for x in lists), dtype=np.int64, count=len(lists))
    off = np.zeros(len(lists) + 1, dtype=np.int64); np.cumsum(lengths, out=off[1:])
    flat = np.fromiter((i for x in lists for i in x), dtype=np.int32, count=int(off[-1]))
    return torch.from_numpy(flat), torch.from_numpy(off)


if not CACHE.exists():
    train_groups = [g for g in range(N) if split[g] == "train"]
    local = {g: i for i, g in enumerate(train_groups)}
    english, eidx, g_eng = [], {}, []
    for g in train_groups:
        ids = []
        for e in dict.fromkeys(G_eng[g]):
            if e not in eidx:
                eidx[e] = len(english); english.append(e)
            ids.append(eidx[e])
        g_eng.append(ids)
    clusters = collections.defaultdict(list)
    for g in train_groups:
        if G_verse[g]:
            clusters[G_verse[g]].append(local[g])
    clusters = [c for _, c in sorted(clusters.items()) if len({G_lang[train_groups[i]] for i in c}) >= 2]
    toks = lambda texts: [x for i in range(0, len(texts), 4096) for x in tok(texts[i:i + 4096], add_special_tokens=False)["input_ids"]]
    s_ids, s_off = ragged(toks([G_text[g] for g in train_groups]))
    e_ids, e_off = ragged(toks(english))
    ge, ge_off = ragged(g_eng)
    cl, cl_off = ragged(clusters)
    lang_codes = [c for c, _ in LANGS.values()]
    torch.save({"langs": lang_codes, "src_ids": s_ids, "src_off": s_off, "eng_ids": e_ids, "eng_off": e_off,
                "group_lang": torch.tensor([lang_codes.index(G_lang[g]) for g in train_groups], dtype=torch.int16),
                "group_synthetic": torch.tensor([G_synth[g] for g in train_groups]),
                "group_eng": ge, "group_eng_off": ge_off, "verse": cl, "verse_off": cl_off}, CACHE)
cache = torch.load(CACHE, weights_only=False)
print(f"train: {len(cache['group_lang']):,} source texts, {int(cache['group_eng_off'][-1]):,} English pairs, "
      f"{len(cache['verse_off']) - 1:,} verses in 2+ non-English languages")

## 7. Batches

An example is a (source, target) pair of texts: a source text and one of its English versions (either
direction), or the same verse in two non-English languages (either direction).

In [ ]:
SRC, ENGK = 0, 1      # text kinds: a source-language text or an English text


class EpochSampler:
    def __init__(self, cache, lang_ids, eng_id):
        self.c, self.lang_ids, self.eng_id = cache, lang_ids, eng_id
        self.length = {SRC: (cache["src_off"][1:] - cache["src_off"][:-1]).numpy(),
                       ENGK: (cache["eng_off"][1:] - cache["eng_off"][:-1]).numpy()}
        self.g_lang = cache["group_lang"].numpy().astype(np.int64)
        self.g_synth = cache["group_synthetic"].numpy()
        self.ge, self.ge_off = cache["group_eng"].numpy(), cache["group_eng_off"].numpy()
        self.vs, self.vs_off = cache["verse"].numpy(), cache["verse_off"].numpy()
        n = np.bincount(self.g_lang, minlength=len(lang_ids)).astype(float)
        self.lang_p = n ** ALPHA / (n ** ALPHA).sum()
        self.lang_groups = [np.flatnonzero(self.g_lang == l) for l in range(len(lang_ids))]

    def epoch(self, e):
        rng = np.random.default_rng(SEED + e)
        rows = []                                         # (source kind, source id, target kind, target id)
        for l, groups in enumerate(self.lang_groups):
            if len(groups) == 0:
                continue
            draws = int(round(len(self.g_lang) * self.lang_p[l]))
            w = np.where(self.g_synth[groups], SYNTHETIC_WEIGHT, 1.0)
            for g in rng.choice(groups, size=draws, replace=draws > len(groups), p=w / w.sum()):
                engs = self.ge[self.ge_off[g]:self.ge_off[g + 1]]
                for en in rng.choice(engs, size=min(K_ENGLISH, len(engs)), replace=False):
                    rows += [(SRC, g, ENGK, en), (ENGK, en, SRC, g)]
        n_eng = len(rows)
        for v in range(len(self.vs_off) - 1):
            by_lang = collections.defaultdict(list)
            for g in self.vs[self.vs_off[v]:self.vs_off[v + 1]]:
                by_lang[self.g_lang[g]].append(g)
            pairs = list(itertools.combinations(sorted(by_lang), 2))
            for k in rng.choice(len(pairs), size=min(XX_PAIRS_PER_VERSE, len(pairs)), replace=False):
                a, b = pairs[k]
                ga, gb = rng.choice(by_lang[a]), rng.choice(by_lang[b])
                rows += [(SRC, ga, SRC, gb), (SRC, gb, SRC, ga)]
        rows = np.array(rows, dtype=np.int64)
        def lens(kind_col, id_col):
            return np.where(rows[:, kind_col] == SRC, self.length[SRC][np.where(rows[:, kind_col] == SRC, rows[:, id_col], 0)],
                            self.length[ENGK][np.where(rows[:, kind_col] == ENGK, rows[:, id_col], 0)])
        need = np.maximum(lens(0, 1), lens(2, 3)) + 2
        batches, lo = [], 0
        for L in BUCKETS:
            idx = np.flatnonzero((need > lo) & (need <= L)); rng.shuffle(idx)
            b = max(1, TOKENS_PER_BATCH // L)
            batches += [(L, rows[idx[i:i + b]]) for i in range(0, len(idx), b)]
            lo = L
        stats = {"english_pairs": n_eng, "other_pairs": len(rows) - n_eng, "skipped": int((need > BUCKETS[-1]).sum())}
        return [batches[i] for i in rng.permutation(len(batches))], len(rows), stats

    def text(self, kind, i):
        c = self.c
        if kind == SRC:
            return c["src_ids"][c["src_off"][i]:c["src_off"][i + 1]].long(), self.lang_ids[self.g_lang[i]]
        return c["eng_ids"][c["eng_off"][i]:c["eng_off"][i + 1]].long(), self.eng_id

    def tensors(self, L, rows, pad_id, eos_id, start_id):
        B = len(rows)
        src = torch.full((B, L), pad_id, dtype=torch.long); lab = torch.full((B, L), -100, dtype=torch.long)
        for r, (sk, si, tk, ti) in enumerate(rows):
            x, xl = self.text(sk, si); y, yl = self.text(tk, ti)
            s = torch.cat([torch.tensor([xl]), x, torch.tensor([eos_id])]); t = torch.cat([torch.tensor([yl]), y, torch.tensor([eos_id])])
            src[r, :len(s)], lab[r, :len(t)] = s, t
        dec = torch.cat([torch.full((B, 1), start_id), lab.masked_fill(lab == -100, pad_id)[:, :-1]], dim=1)
        return src, dec, lab

## 8. Model, optimizer and resume

In [ ]:
from transformers import AutoModelForSeq2SeqLM
from transformers.optimization import Adafactor

NEW_CODES = [lc for lc, _ in LANGS.values() if tok.convert_tokens_to_ids(lc) in (None, tok.unk_token_id)]
tok.add_tokens(NEW_CODES, special_tokens=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, attn_implementation="sdpa")
model.resize_token_embeddings(len(tok), mean_resizing=False)
with torch.no_grad():
    emb = model.get_input_embeddings().weight
    start = emb[[tok.convert_tokens_to_ids(c) for c in INIT_FROM]].mean(0)
    for c in NEW_CODES:
        emb[tok.convert_tokens_to_ids(c)] = start          # the output layer shares these weights
model.config.use_cache = False
model.to(DEVICE).train()
lang_ids = [tok.convert_tokens_to_ids(c) for c in cache["langs"]]
eng_id = tok.convert_tokens_to_ids(ENG)
sampler = EpochSampler(cache, lang_ids, eng_id)
opt = Adafactor(model.parameters(), lr=LR, scale_parameter=False, relative_step=False, warmup_init=False)
print(f"new codes {NEW_CODES}; {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")
print("language sampling (English pairs):", ", ".join(f"{CODE2NAME[c]} {p:.0%}" for c, p in zip(cache["langs"], sampler.lang_p)))

STATE, BEST = OUT / "last" / "state.pt", OUT / "best"
progress = {"step": 0, "epoch": 0, "done_in_epoch": 0, "elapsed": 0.0, "best_dev": -1.0, "best_step": 0, "bad_evals": 0, "stopped": False}
if STATE.exists():
    st = torch.load(STATE, map_location="cpu", weights_only=False)
    model.load_state_dict(st["model"]); opt.load_state_dict(st["optimizer"]); progress.update(st["progress"])
    print(f"resuming at step {progress['step']:,} ({progress['elapsed'] / 3600:.2f} h so far, best dev chrF++ {progress['best_dev']:.2f} "
          f"at step {progress['best_step']:,}, {progress['bad_evals']} evaluation(s) without improvement)")

## 9. Evaluation functions

chrF++ is sacreBLEU chrF with word bigrams (`word_order=2`). Model selection uses the mean dev chrF++ over all
42 directions, so every direction counts equally.

In [ ]:
from sacrebleu.metrics import BLEU, CHRF
from sacrebleu.significance import _bootstrap_resample


@torch.no_grad()
def translate_items(m, items, beam):
    m.eval(); m.config.use_cache = True
    order = sorted(range(len(items)), key=lambda i: len(items[i]["src"]))
    out = [None] * len(items)
    for b in range(0, len(order), EVAL_BATCH):
        by_tgt = collections.defaultdict(list)               # one forced target code per generate call
        for i in order[b:b + EVAL_BATCH]:
            it = items[i]
            ids = tok(it["src"], add_special_tokens=False)["input_ids"][:BUCKETS[-1] - 2]
            by_tgt[it["tgt_lang"]].append((i, [tok.convert_tokens_to_ids(it["src_lang"])] + ids + [tok.eos_token_id]))
        for tgt, rows in by_tgt.items():
            W = max(len(r) for _, r in rows)
            x = torch.full((len(rows), W), tok.pad_token_id, dtype=torch.long)
            for k, (_, r) in enumerate(rows):
                x[k, :len(r)] = torch.tensor(r)
            x = x.to(DEVICE)
            with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
                y = m.generate(input_ids=x, attention_mask=(x != tok.pad_token_id), forced_bos_token_id=tok.convert_tokens_to_ids(tgt),
                               num_beams=beam, max_new_tokens=int(1.5 * W + 10))
            for (i, _), h in zip(rows, tok.batch_decode(y, skip_special_tokens=True)):
                out[i] = h.strip()
    m.config.use_cache = False; m.train()
    return out


def score(hyps, refs, n_boot=1):
    # chrF++ and BLEU with sacreBLEU's bootstrap. The 95% half-width is computed here the way sacreBLEU does it,
    # because its own averaging step fails on some Python versions when resampled scores are exactly 0.
    r = {"n": len(hyps)}
    for name, metric in (("chrf", CHRF(word_order=2)), ("bleu", BLEU())):
        stats = metric._extract_corpus_statistics(hyps, [refs])
        r[name] = round(float(metric._aggregate_and_compute(stats).score), 2)
        if n_boot > 1:
            metric.n_bootstrap = n_boot
            metric.seed, bs = _bootstrap_resample(stats, metric, n_boot)
            raw = np.sort([float(x.score) for x in bs]); lo = len(raw) // 40
            r[name + "_ci"] = round(0.5 * float(raw[len(raw) - lo - 1] - raw[lo]), 2)
        r["sig_" + name] = str(metric.get_signature())
    return r


def dname(a, b):
    return f"{CODE2NAME[a]}->{CODE2NAME[b]}"


@torch.no_grad()
def dev_eval(step):
    hyps = translate_items(model, dev_sample, beam=1)
    by = collections.defaultdict(list)
    for i, it in enumerate(dev_sample):
        by[(it["src_lang"], it["tgt_lang"])].append(i)
    res = {dname(a, b): CHRF(word_order=2).corpus_score([hyps[i] for i in idx], [[dev_sample[i]["ref"] for i in idx]]).score
           for (a, b), idx in sorted(by.items())}
    mean = float(np.mean(list(res.values()))) if res else 0.0
    log = OUT / "dev_log.csv"
    if not log.exists():
        log.write_text("step,mean_chrf," + ",".join(res) + "\n")
    with open(log, "a") as f:
        f.write(f"{step},{mean:.2f}," + ",".join(f"{v:.2f}" for v in res.values()) + "\n")
    return mean, res

## 10. Step-0 reference point

Scores the untrained starting model on the dev sample (once, before training) and saves it as the first
*best* checkpoint, so training has to beat it. Directions involving a Tani language are expected near zero;
Hindi ↔ English shows what NLLB-200 already knows.

In [ ]:
if progress["step"] == 0 and not (OUT / "dev_log.csv").exists():
    mean0, res0 = dev_eval(0)
    progress["best_dev"], progress["best_step"] = mean0, 0
    model.config.use_cache = True; model.save_pretrained(BEST); tok.save_pretrained(BEST); model.config.use_cache = False
    print(f"step 0: mean dev chrF++ {mean0:.2f} over {len(res0)} directions (saved as the starting best checkpoint)")
    print("  highest:", ", ".join(f"{k} {v:.1f}" for k, v in sorted(res0.items(), key=lambda x: -x[1])[:5]))

## 11. Train

Every `EVAL_EVERY_STEPS` steps it scores the dev sample and keeps the best checkpoint in `OUT/best`. Training
stops when mean dev chrF++ has not improved (by more than `MIN_DELTA`) for `PATIENCE` evaluations in a row.
Interrupting the kernel saves a checkpoint first; running the cells again resumes, and does nothing more once
early stopping has triggered.

In [ ]:
import math, os
import torch.nn.functional as F

TRAIN_LOG = OUT / "train_log.csv"


def lr_at(step):
    step = max(step, 1)
    return LR * min(step / WARMUP, math.sqrt(WARMUP / step))


def save_state(t_start):
    STATE.parent.mkdir(parents=True, exist_ok=True)
    p = dict(progress, elapsed=progress["elapsed"] + time.time() - t_start)
    torch.save({"model": model.state_dict(), "optimizer": opt.state_dict(), "progress": p}, STATE.with_suffix(".tmp"))
    os.replace(STATE.with_suffix(".tmp"), STATE)


def train():
    pad_id, eos_id, start_id = tok.pad_token_id, tok.eos_token_id, model.config.decoder_start_token_id
    t_start = last_save = window = time.time()
    tokens, losses, projected = 0, [], False
    if not TRAIN_LOG.exists():
        TRAIN_LOG.write_text("step,epoch,loss,lr,tokens_per_s,hours\n")
    try:
        while True:
            batches, n_rows, stats = sampler.epoch(progress["epoch"])
            per_epoch = len(batches) // ACCUM
            if progress["done_in_epoch"] == 0:
                print(f"epoch {progress['epoch']}: {n_rows:,} examples ({stats['english_pairs']:,} with English, "
                      f"{stats['other_pairs']:,} between other languages), {per_epoch:,} steps, {stats['skipped']:,} over {BUCKETS[-1]} tokens skipped")
            for bi in range(progress["done_in_epoch"] * ACCUM, per_epoch * ACCUM):
                L, rows = batches[bi]
                src, dec, lab = (t.to(DEVICE, non_blocking=True) for t in sampler.tensors(L, rows, pad_id, eos_id, start_id))
                with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
                    logits = model(input_ids=src, attention_mask=(src != pad_id), decoder_input_ids=dec).logits
                loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), lab.view(-1), ignore_index=-100, label_smoothing=LABEL_SMOOTHING)
                (loss / ACCUM).backward()
                losses.append(loss.item()); tokens += int((lab != -100).sum())
                if (bi + 1) % ACCUM:
                    continue
                progress["step"] += 1; progress["done_in_epoch"] = (bi + 1) // ACCUM
                for g in opt.param_groups:
                    g["lr"] = lr_at(progress["step"])
                opt.step(); opt.zero_grad(set_to_none=True)
                step, used = progress["step"], progress["elapsed"] + time.time() - t_start
                if step % LOG_EVERY == 0:
                    dt = time.time() - window
                    with open(TRAIN_LOG, "a") as f:
                        f.write(f"{step},{progress['epoch']},{np.mean(losses):.4f},{lr_at(step):.3e},{tokens / dt:.0f},{used / 3600:.3f}\n")
                    print(f"step {step:,}  epoch {progress['epoch']} ({progress['done_in_epoch']:,}/{per_epoch:,})  loss {np.mean(losses):.3f}  "
                          f"{tokens / dt:,.0f} tok/s  {used / 3600:.2f} h")
                    if not projected and step >= 3 * LOG_EVERY:
                        sec = dt / LOG_EVERY
                        print(f"  speed {sec:.2f} s/step: one epoch ({per_epoch:,} steps) takes about {per_epoch * sec / 3600:.1f} h; "
                              f"a dev evaluation every {EVAL_EVERY_STEPS:,} steps = every {EVAL_EVERY_STEPS * sec / 3600:.1f} h")
                        projected = True
                    tokens, losses, window = 0, [], time.time()
                if step % EVAL_EVERY_STEPS == 0:
                    mean, _ = dev_eval(step)
                    better = mean > progress["best_dev"] + MIN_DELTA
                    if better:
                        progress["best_dev"], progress["best_step"], progress["bad_evals"] = mean, step, 0
                        model.config.use_cache = True; model.save_pretrained(BEST); tok.save_pretrained(BEST); model.config.use_cache = False
                    else:
                        progress["bad_evals"] += 1
                    print(f"  dev chrF++ {mean:.2f}" + ("  (best so far, saved)" if better else
                          f"  (no improvement {progress['bad_evals']}/{PATIENCE}; best {progress['best_dev']:.2f} at step {progress['best_step']:,})"))
                    window = time.time()
                    if progress["bad_evals"] >= PATIENCE:
                        progress["stopped"] = True
                        save_state(t_start); return True
                if time.time() - last_save > SAVE_MINUTES * 60:
                    save_state(t_start); last_save = time.time()
            progress["epoch"] += 1; progress["done_in_epoch"] = 0
    except KeyboardInterrupt:
        save_state(t_start); print("interrupted; checkpoint saved"); return False


if progress["stopped"]:
    print(f"early stopping already reached; best checkpoint at step {progress['best_step']:,}")
    finished = True
else:
    finished = train()
print(f"early stopping: no dev improvement for {PATIENCE} evaluations; best checkpoint at step {progress['best_step']:,}"
      if finished else "interrupted; run all cells again to resume")

## 12. Final evaluation (run once)

Loads the best dev checkpoint and translates the test items of every direction with beam search (at most
`TEST_MAX_PER_DIRECTION` clean items per direction, sampled with the seed). Reports chrF++ and BLEU with 95%
bootstrap confidence intervals on clean items (headline) and on all items, and by domain for the English
directions.

In [ ]:
import pandas as pd

final = AutoModelForSeq2SeqLM.from_pretrained(BEST, attn_implementation="sdpa").to(DEVICE)
trng = np.random.default_rng(SEED + 1)


def test_items(directions):
    items = []
    for a, b in directions:
        pool = [it for it in eval_items if it["split"] == "test" and it["src_lang"] == a and it["tgt_lang"] == b]
        clean = [k for k, it in enumerate(pool) if it["clean"]]
        if TEST_MAX_PER_DIRECTION and len(clean) > TEST_MAX_PER_DIRECTION:
            keep = set(trng.choice(clean, TEST_MAX_PER_DIRECTION, replace=False).tolist())
            pool = [it for k, it in enumerate(pool) if not it["clean"] or k in keep]
        items += [dict(it) for it in pool]
    return items


def evaluate(m, items, label):
    for it, h in zip(items, translate_items(m, items, beam=TEST_BEAM)):
        it["hyp"] = h
    df, rows = pd.DataFrame(items), []
    for subset, keys in (("clean", ["src_lang", "tgt_lang"]), ("all", ["src_lang", "tgt_lang"]), ("clean", ["src_lang", "tgt_lang", "domain"])):
        for key, grp in df.groupby(keys):
            if len(keys) == 3 and ENG not in key[:2]:
                continue
            g = grp[grp["clean"]] if subset == "clean" else grp
            if len(g):
                rows.append({"model": label, "subset": subset, "source": CODE2NAME[key[0]], "target": CODE2NAME[key[1]],
                             "domain": key[2] if len(keys) == 3 else "all", **score(list(g["hyp"]), list(g["ref"]), BOOTSTRAP)})
    df.to_csv(OUT / f"predictions_{label.replace(' ', '_')}.csv", index=False)
    return rows


results = evaluate(final, test_items(DIRECTIONS), "fine-tuned")
if BASELINE_HINDI:
    base = AutoModelForSeq2SeqLM.from_pretrained(MODEL, attn_implementation="sdpa").to(DEVICE)
    results += evaluate(base, test_items([("hin_Deva", ENG), (ENG, "hin_Deva")]), "untouched NLLB-200")
    del base; torch.cuda.empty_cache()
res = pd.DataFrame(results)
res.to_csv(OUT / "results.csv", index=False)
(OUT / "results.json").write_text(json.dumps({"best_step": progress["best_step"], "best_dev_chrf": progress["best_dev"],
                                              "steps": progress["step"], "hours": progress["elapsed"] / 3600, "results": results}, indent=2))
head = res[(res["model"] == "fine-tuned") & (res["subset"] == "clean") & (res["domain"] == "all")]
print(f"best checkpoint: step {progress['best_step']:,} (dev chrF++ {progress['best_dev']:.2f}); {len(head)} directions scored")
print(head[["source", "target", "n", "chrf", "chrf_ci", "bleu", "bleu_ci"]].to_string(index=False))

## 13. Figures

In [ ]:
import matplotlib.pyplot as plt

NAMES = [CODE2NAME[c] for c in ALL_CODES]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
tl = pd.read_csv(TRAIN_LOG)
ax[0].plot(tl["step"], tl["loss"], lw=1); ax[0].set(xlabel="optimizer step", ylabel="training loss (label-smoothed)", title="Training loss")
dl = pd.read_csv(OUT / "dev_log.csv")
for c in dl.columns[2:]:
    ax[1].plot(dl["step"], dl[c], lw=0.8, alpha=0.35)
ax[1].plot(dl["step"], dl["mean_chrf"], lw=2.5, color="black", label="mean of 42 directions")
ax[1].set(xlabel="optimizer step", ylabel="dev chrF++ (greedy)", title="Dev chrF++ per direction"); ax[1].legend(fontsize=8)
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / "curves.png", dpi=150); plt.show()

M = np.full((len(NAMES), len(NAMES)), np.nan)
for r in head.itertuples():
    M[NAMES.index(r.source), NAMES.index(r.target)] = r.chrf
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(M, cmap="viridis", vmin=0, vmax=max(1, np.nanmax(M)))
for i in range(len(NAMES)):
    for j in range(len(NAMES)):
        if not np.isnan(M[i, j]):
            ax.text(j, i, f"{M[i, j]:.1f}", ha="center", va="center", fontsize=8, color="white" if M[i, j] < np.nanmax(M) * 0.6 else "black")
ax.set_xticks(range(len(NAMES))); ax.set_xticklabels(NAMES, rotation=45, ha="right"); ax.set_yticks(range(len(NAMES))); ax.set_yticklabels(NAMES)
ax.set(xlabel="target", ylabel="source", title="Test chrF++ (clean items), source → target")
fig.colorbar(im, ax=ax, shrink=0.8); plt.tight_layout(); plt.savefig(OUT / "test_chrf_matrix.png", dpi=150); plt.show()

## 14. Report

In [ ]:
def matrix_md(metric):
    lines = ["| source \\ target | " + " | ".join(NAMES) + " |", "|---|" + "---:|" * len(NAMES)]
    for s in NAMES:
        cells = []
        for t in NAMES:
            r = head[(head["source"] == s) & (head["target"] == t)]
            cells.append("—" if s == t else f"{r[metric].iloc[0]:.1f}" if len(r) else "")
        lines.append(f"| {s} | " + " | ".join(cells) + " |")
    return "\n".join(lines)


def table_md(df, first="model"):
    lines = [f"| {first.capitalize()} | Source | Target | n | chrF++ | BLEU |", "|---|---|---|---:|---:|---:|"]
    for r in df.itertuples():
        lines.append(f"| {getattr(r, first)} | {r.source} | {r.target} | {r.n:,} | {r.chrf:.1f} ± {r.chrf_ci:.1f} | {r.bleu:.1f} ± {r.bleu_ci:.1f} |")
    return "\n".join(lines)


env = json.loads(env_path.read_text())
xx = head[(head["source"] != "English") & (head["target"] != "English")]
ex = head[(head["source"] == "English") | (head["target"] == "English")]
report = f"""# NLLB-200 600M fine-tuned for 42 translation directions: results

- Model selected at step {progress['best_step']:,} (mean dev chrF++ over 42 directions {progress['best_dev']:.2f}); training stopped at step {progress['step']:,} after {PATIENCE} dev evaluations without improvement ({progress['elapsed'] / 3600:.1f} h on {env['gpu']}).
- Split: 90/5/5 by Bible chapter (shared across all languages) and by English-linked groups.
- Test scores on **clean** items (no English reference seen in training, no near-duplicate source in training), beam {TEST_BEAM}, at most {TEST_MAX_PER_DIRECTION} items per direction, ± = 95% bootstrap CI ({BOOTSTRAP} resamples).
- Mean chrF++: all 42 directions {head['chrf'].mean():.1f}; with English {ex['chrf'].mean():.1f}; between other languages (Bible only) {xx['chrf'].mean():.1f}.
- Signatures: `{res['sig_chrf'].iloc[0]}`, `{res['sig_bleu'].iloc[0]}`.

## chrF++ matrix (test, clean)

{matrix_md('chrf')}

## BLEU matrix (test, clean)

{matrix_md('bleu')}

## Every direction with confidence intervals

{table_md(res[(res['subset'] == 'clean') & (res['domain'] == 'all')])}

## English directions by domain (test, clean)

{table_md(res[(res['model'] == 'fine-tuned') & (res['subset'] == 'clean') & (res['domain'] != 'all')], first='domain')}

## All items, including flagged

{table_md(res[(res['model'] == 'fine-tuned') & (res['subset'] == 'all')])}
"""
(OUT / "REPORT.md").write_text(report)
print(report)